# Threshold-Based VOI Segmentation

In [169]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_fill_holes

%matplotlib inline

## Load Data

In [171]:
from scipy.ndimage import zoom

scan_path = '/Volumes/Extreme Pro/UCSD_SSD/ChinaData/3D-034/2nd/CEUS-26285-2.nii.gz'
voi_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/necrosis_testing/p34/v2/big_voi.nii.gz'
paramap_folder = '/Users/samantha/Desktop/ultrasound lab stuff/china data/necrosis_testing/p34/v2'
output_folder = '/Users/samantha/Desktop/ultrasound lab stuff/china data/necrosis_testing/p34/v2'

scan_nii = nib.load(scan_path)
scan = scan_nii.get_fdata()
affine = scan_nii.affine

voi = nib.load(voi_path).get_fdata()
voi_mask = voi > 0

ttp = np.load(f'{paramap_folder}/TP_full_TIC_numerical.npy')

# Temporal variance (resample to match VOI/paramap resolution)
temporal_var = np.std(scan, axis=3)
if temporal_var.shape != voi_mask.shape:
    scale = np.array(voi_mask.shape) / np.array(temporal_var.shape)
    temporal_var = zoom(temporal_var, scale, order=1)
    print(f"Resampled temporal_var: {temporal_var.shape}")

print(f"Scan shape: {scan.shape}")
print(f"VOI voxels: {np.sum(voi_mask):,}")

Scan shape: (240, 205, 128, 210)
VOI voxels: 211,395


## Set Thresholds

In [172]:
ttp_in_voi = ttp[voi_mask & ~np.isnan(ttp)]
var_in_voi = temporal_var[voi_mask]

# Percentiles
percentile_ttp = 75  # earliest X%
percentile_var = 72  # top X% variance

thresh_ttp = np.percentile(ttp_in_voi, percentile_ttp)
thresh_var = np.percentile(var_in_voi, 100 - percentile_var)

print(f"Thresholds:")
print(f"  TTP <= {thresh_ttp:.2f}s (earliest {percentile_ttp}%)")
print(f"  Variance >= {thresh_var:.2f} (top {percentile_var}%)")

Thresholds:
  TTP <= 47.21s (earliest 75%)
  Variance >= 11.28 (top 72%)


## Create Masks

In [173]:
mask_ttp = voi_mask & ~np.isnan(ttp) & (ttp <= thresh_ttp)
mask_ttp = binary_fill_holes(mask_ttp) & voi_mask

mask_var = voi_mask & (temporal_var >= thresh_var)
mask_var = binary_fill_holes(mask_var) & voi_mask

mask_combined = mask_ttp & mask_var

print(f"Original VOI: {np.sum(voi_mask):,}")
print(f"TTP: {np.sum(mask_ttp):,}")
print(f"Variance: {np.sum(mask_var):,}")
print(f"Combined (TTP & Var): {np.sum(mask_combined):,}")

Original VOI: 211,395
TTP: 155,472
Variance: 152,266
Combined (TTP & Var): 117,128


## Visualize

In [167]:
import napari

t = 30  # time frame for background

# Resample scan frame to match VOI/mask grid
scale = np.array(voi_mask.shape) / np.array(scan.shape[:3])
scan_frame = zoom(scan[:,:,:,t], scale, order=1)

# Transpose (2,1,0) so axial is scrollable and orientation matches QuantUS GUI
viewer = napari.Viewer()
viewer.add_image(scan_frame.transpose(2, 1, 0), name=f'Scan (t={t})', colormap='gray')
viewer.add_labels(voi_mask.astype(np.uint8).transpose(2, 1, 0), name='Original VOI', opacity=0.15)
viewer.add_labels(mask_ttp.astype(np.uint8).transpose(2, 1, 0), name='TTP mask', opacity=0.25, visible=False)
viewer.add_labels(mask_var.astype(np.uint8).transpose(2, 1, 0), name='Variance mask', opacity=0.25, visible=False)
viewer.add_labels(mask_combined.astype(np.uint8).transpose(2, 1, 0), name='Combined mask', opacity=0.3)

<Labels layer 'Combined mask' at 0x1118ef0d0>

## Compute TIC Stats (AUC, PE) for Each Mask

In [166]:
auc = np.load(f'{paramap_folder}/AUC_full_TIC_numerical.npy')
pe = np.load(f'{paramap_folder}/PE_full_TIC_numerical.npy')

masks = {'TTP': mask_ttp, 'Variance': mask_var, 'Combined': mask_combined}

print(f"{'Mask':<14} {'Voxels':>10} {'AUC mean':>12} {'AUC std':>12} {'PE mean':>12} {'PE std':>12}")
print("-" * 72)

for name, mask in masks.items():
    auc_vals = auc[mask & ~np.isnan(auc)]
    pe_vals = pe[mask & ~np.isnan(pe)]
    
    if len(auc_vals) > 0 and len(pe_vals) > 0:
        print(f"{name:<14} {np.sum(mask):>10,} {np.mean(auc_vals):>12.2f} {np.std(auc_vals):>12.2f} {np.mean(pe_vals):>12.4f} {np.std(pe_vals):>12.4f}")
    else:
        print(f"{name:<14} {np.sum(mask):>10,}  (no paramap overlap)")

Mask               Voxels     AUC mean      AUC std      PE mean       PE std
------------------------------------------------------------------------
TTP               155,472       113.63        75.33       0.8107       0.1782
Variance          152,266       143.57        84.10       0.8335       0.1632
Combined          117,128       127.17        79.58       0.8630       0.1443


## Save Masks

In [161]:
for name, mask in [('ttp', mask_ttp), ('var', mask_var), ('combined', mask_combined)]:
    nii = nib.Nifti1Image(mask.astype(np.uint8), affine)
    nib.save(nii, f'{output_folder}/voi_thresh_{name}.nii.gz')
    print(f"Saved voi_thresh_{name}.nii.gz")

Saved voi_thresh_ttp.nii.gz
Saved voi_thresh_var.nii.gz
Saved voi_thresh_combined.nii.gz
